# 🧬 PANACEE – Notebook 00 : Configuration & Installation
## Projet de prédiction moléculaire par GNN

**Ce notebook effectue :**
1. ✅ Vérification GPU (CUDA)
2. 📦 Installation de toutes les dépendances
3. 📁 Upload des fichiers Python du projet
4. 🔬 Test des imports
5. 📊 Aperçu de l'architecture du projet

---
> **⚠️ Important :** Activer le GPU dans Colab avant de lancer ce notebook !
> `Menu → Exécution → Modifier le type d'exécution → GPU (T4)`

## Étape 1 – Vérification du GPU

In [ ]:
import torch
import platform

print("=" * 60)
print("VÉRIFICATION ENVIRONNEMENT PANACEE")
print("=" * 60)
print(f"Python          : {platform.python_version()}")
print(f"PyTorch         : {torch.__version__}")
print(f"GPU disponible  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM totale     : {mem_total:.1f} GB")
    print("✅ GPU prêt pour l'entraînement !")
else:
    print("⚠️  Aucun GPU détecté – Aller dans Exécution → Modifier le type d'exécution → GPU")

## Étape 2 – Installation des dépendances

Cette cellule installe :
- **PyTorch Geometric** (GNN)
- **RDKit** (cheminformatique)
- **DeepChem** (datasets moléculaires)
- **scikit-learn, scipy** (métriques)

⏱️ Temps estimé : 5-10 minutes sur Colab

In [ ]:
import subprocess, sys

def install(cmd):
    """Lance pip silencieusement et affiche le résultat."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd.split(), capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  ⚠️  Erreur : {result.stderr[-300:]}")
    else:
        print("  ✅ OK")

# PyTorch Geometric + extensions
torch_ver = torch.__version__.split("+")[0]
cuda_ver  = "cu121" if torch.cuda.is_available() else "cpu"

print("\n📦 Installation PyTorch Geometric...")
install(f"pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html -q")
install(f"pip install torch-sparse  -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html -q")
install(f"pip install torch-geometric -q")

print("\n📦 Installation RDKit...")
install("pip install rdkit-pypi -q")

print("\n📦 Installation DeepChem...")
install("pip install deepchem -q")

print("\n📦 Installation des utilitaires...")
install("pip install scikit-learn scipy tqdm matplotlib pandas -q")

print("\n🎉 Installation terminée !")

## Étape 3 – Upload des fichiers du projet

**Fichiers à uploader** (depuis votre ordinateur) :

| Fichier | Rôle |
|---------|------|
| `config.py` | Configuration centrale |
| `graph_builder.py` | SMILES → graphe PyG |
| `data_loaders.py` | Chargement données |
| `gnn_models.py` | 5 architectures GNN |
| `prediction_heads.py` | Têtes de prédiction |
| `losses.py` | Fonctions de perte |
| `metrics.py` | Métriques + monitoring |
| `trainer.py` | Entraîneur unifié |
| `predictor.py` | Inférence |
| `advanced_algorithms.py` | Algorithmes avancés |

**Optionnel :** `pretrain_dataset.pt` (données ZINC pré-traitées, ~1.5 GB)

In [ ]:
import os

# Créer le dossier de travail
os.makedirs("/content/panacee", exist_ok=True)
os.makedirs("/content/panacee/checkpoints/phase1", exist_ok=True)
os.makedirs("/content/panacee/checkpoints/phase2", exist_ok=True)
os.makedirs("/content/panacee/checkpoints/phase3", exist_ok=True)
os.makedirs("/content/panacee/data", exist_ok=True)
os.makedirs("/content/panacee/logs", exist_ok=True)
os.makedirs("/content/panacee/results", exist_ok=True)

# Option A : Upload via l'interface Colab
from google.colab import files

print("📤 Sélectionnez TOUS les fichiers .py du dossier PanaceeColab")
print("(Ctrl+A pour tout sélectionner)")
uploaded = files.upload()

# Déplacer les fichiers vers /content/panacee/
for fname in uploaded.keys():
    src = fname
    dst = f"/content/panacee/{fname}"
    os.rename(src, dst)
    print(f"  ✅ {fname} → {dst}")

print(f"\n📁 Fichiers dans /content/panacee/")
for f in sorted(os.listdir("/content/panacee")):
    print(f"  {f}")

In [ ]:
# Option B : Depuis Google Drive (si vous avez synchronisé le dossier)
# Décommentez et adaptez le chemin si vous préférez cette méthode

# from google.colab import drive
# drive.mount('/content/drive')
# 
# import shutil
# src_dir = "/content/drive/MyDrive/PanaceeColab"  # adaptez ce chemin
# dst_dir = "/content/panacee"
# 
# for f in os.listdir(src_dir):
#     if f.endswith('.py'):
#         shutil.copy(os.path.join(src_dir, f), dst_dir)
#         print(f"  ✅ Copié : {f}")

## Étape 4 – Test des imports

In [ ]:
import sys
sys.path.insert(0, "/content/panacee")
os.chdir("/content/panacee")

errors = []

# Test chaque module
modules_to_test = [
    ("config",              "DEVICE, HIDDEN_DIM"),
    ("graph_builder",       "smiles_to_graph"),
    ("data_loaders",        "PretrainDataset, ToxicityDataset"),
    ("gnn_models",          "build_encoder"),
    ("prediction_heads",    "ToxicityClassifier, MultiPropertyPredictor"),
    ("losses",              "MultiTaskBCELoss, MultiPropertyLoss"),
    ("metrics",             "EarlyStopping, TrainingHistory"),
    ("trainer",             "train_phase1, train_phase2, train_phase3"),
    ("predictor",           "PanaceePredictor"),
    ("advanced_algorithms", "MCTSCombinationSearch, BayesianOptimizer"),
]

print("=" * 55)
print("TEST DES IMPORTS")
print("=" * 55)
for mod, symbols in modules_to_test:
    try:
        exec(f"from {mod} import {symbols}")
        print(f"  ✅ {mod:<25} [{symbols[:30]}...]")
    except Exception as e:
        print(f"  ❌ {mod:<25} ERREUR: {e}")
        errors.append((mod, str(e)))

print()
if errors:
    print(f"⚠️  {len(errors)} erreur(s) détectée(s) :")
    for mod, err in errors:
        print(f"   - {mod}: {err}")
else:
    print("✅ Tous les imports fonctionnent !")

## Étape 5 – Test rapide : SMILES → Graphe → GNN

In [ ]:
from config import DEVICE
from graph_builder import smiles_to_graph
from gnn_models import build_encoder, count_parameters
from torch_geometric.data import Batch

# Molécules test : Paracétamol, Aspirine, Caféine
test_smiles = [
    "CC(=O)Nc1ccc(O)cc1",       # Paracétamol
    "CC(=O)Oc1ccccc1C(=O)O",    # Aspirine
    "Cn1c(=O)c2c(ncn2C)n(c1=O)C",  # Caféine
]

print("Test SMILES → Graphe PyTorch Geometric")
print("-" * 50)
graphs = []
for smi in test_smiles:
    g = smiles_to_graph(smi)
    if g is not None:
        graphs.append(g)
        print(f"  ✅ {smi[:40]:<40} | {g.num_nodes} atomes | {g.num_edges//2} liaisons")
    else:
        print(f"  ❌ SMILES invalide : {smi}")

print()
print("Test passage dans l'encodeur GNN (AttFP)")
print("-" * 50)
encoder = build_encoder(arch="attfp").to(DEVICE)
print(f"  Paramètres totaux : {count_parameters(encoder):,}")

batch = Batch.from_data_list(graphs).to(DEVICE)
with torch.no_grad():
    emb = encoder(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

print(f"  Entrée  : {batch.num_graphs} graphes | {batch.num_nodes} atomes")
print(f"  Sortie  : {emb.shape}  (batch × embedding_dim)")
print(f"  Device  : {emb.device}")
print()
print("✅ Pipeline complet fonctionnel !")

## Récapitulatif des architectures disponibles

In [ ]:
from gnn_models import build_encoder, count_parameters

print("=" * 65)
print("ARCHITECTURES GNN DISPONIBLES")
print("=" * 65)
print(f"{'Nom':<10} {'Paramètres':>15}  Description")
print("-" * 65)

architectures = {
    "mpnn"  : "MPNN – Message Passing Neural Network (Gilmer 2017)",
    "attfp" : "AttFP – Attentive FP (Xiong 2020 JACS) [RECOMMANDÉ]",
    "gin"   : "GIN – Graph Isomorphism Network (Xu 2019 + Hu 2020)",
    "pna"   : "PNA – Principal Neighbourhood Aggregation (Corso 2020)",
    "gps"   : "GPS – Graph Positional+Structural (Rampásek 2022 NeurIPS)",
}

for arch, desc in architectures.items():
    enc = build_encoder(arch=arch)
    n   = count_parameters(enc)
    print(f"  {arch:<10} {n:>12,}  {desc}")

print()
print("💡 Pour changer d'architecture : modifiez GNN_ARCHITECTURE dans config.py")

## ➡️ Étapes suivantes

Une fois ce notebook exécuté avec succès :

1. **`Notebook_01_Phase1_Pretrain.ipynb`** — Pré-entraînement sur ZINC (Masked Graph Modeling)
2. **`Notebook_02_Phase2_Toxicity.ipynb`** — Fine-tuning Tox21 (classification de toxicité)
3. **`Notebook_03_Phase3_MultiProp.ipynb`** — Multi-propriétés + analyse combinatoire

---
*Consultez `GUIDE_COMPLET.md` pour les instructions détaillées.*